# CLIP4CAD Training with AutoBrep-Style Encoder

## Architecture Summary

This notebook trains a BRep encoder following **AutoBrep's design principles**:

| Component | AutoBrep (Generation) | Our Encoder (Embedding) |
|-----------|----------------------|------------------------|
| Layers | 16 | **6** (encoder needs less) |
| Hidden dim | 2048 | **512** (scaled for embedding) |
| Heads | 32 | **8** |
| FFN mult | 4x | 4x |
| Dropout | 0.1 | 0.1 |
| Message passing | None | None |
| Codebook | None | None |

## Key Insight

AutoBrep proves that a **transformer-only approach works** on our FSQ latent representation.
No explicit message passing or topology encoding is needed - the transformer learns it.

## Training: Stage 0 Only

This is a simplified training loop focused on:
- **Anchor BRep to PC**: Use pre-trained ShapeLLM as anchor
- **Contrastive loss**: InfoNCE between BRep and PC embeddings
- **Simple and effective**: No codebook, no text, just BRep-PC alignment

In [ ]:
# Cell 1: Imports and Setup
import sys
sys.path.insert(0, '..')

import os
import gc
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
import numpy as np
from pathlib import Path

# Clean memory
gc.collect()
torch.cuda.empty_cache()

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Configuration
# ═══════════════════════════════════════════════════════════════════════════════
# DATA PATHS
# ═══════════════════════════════════════════════════════════════════════════════
DATA_ROOT = Path("d:/Defect_Det/MMCAD/data")
EMBEDDINGS_DIR = DATA_ROOT / "embeddings"
ALIGNED_DIR = DATA_ROOT / "aligned"

# PC file (ShapeLLM features)
PC_FILE = Path("c:/Users/User/Desktop/pc_embeddings_full.h5")

# Text file (for later stages, not used in Stage 0)
TEXT_FILE = Path("c:/Users/User/Desktop/text_embeddings.h5")

# Output
OUTPUT_DIR = Path("../outputs/autobrep")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# TRAINING HYPERPARAMETERS
# ═══════════════════════════════════════════════════════════════════════════════
BATCH_SIZE = 128          # Larger batch for contrastive learning
NUM_EPOCHS = 20           # Stage 0 training
LEARNING_RATE = 5e-4      # Higher LR for initial training
MAX_GRAD_NORM = 1.0

# Model configuration
D_MODEL = 512             # Hidden dimension
NUM_LAYERS = 6            # Transformer layers
NUM_HEADS = 8             # Attention heads
D_PROJ = 128              # Output projection dimension

print("Configuration:")
print(f"  Data root: {DATA_ROOT}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Model: d={D_MODEL}, layers={NUM_LAYERS}, heads={NUM_HEADS}")
print(f"  Output: {OUTPUT_DIR}")

In [ ]:
# Cell 3: Import Dataset
from clip4cad.data.gfa_dataset import GFAMappedDataset, gfa_collate_fn

print("Using GFAMappedDataset with use_autobrep=True")

In [ ]:
# Cell 4: Load Datasets
gc.collect()
torch.cuda.empty_cache()

print("Loading datasets with AutoBrep topology...")
print("=" * 60)

# For Stage 0, we don't need text embeddings loaded to RAM
MAX_TRAIN_SAMPLES = 111000  # Limit for memory

# Train dataset - LOAD TO RAM
print(f"\n[1/2] Loading TRAIN dataset (max {MAX_TRAIN_SAMPLES:,} samples)...")
train_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="train",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=True,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
    max_samples=MAX_TRAIN_SAMPLES,
)
print(f"Train: {len(train_dataset):,} samples")

# Val dataset - ON DISK (smaller, less critical)
print("\n[2/2] Loading VAL dataset...")
val_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="val",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    mapping_dir=str(ALIGNED_DIR),
    load_to_memory=False,
    use_live_text=False,
    use_autobrep=True,
    autobrep_dir=str(EMBEDDINGS_DIR),
)
print(f"Val: {len(val_dataset):,} samples")

print("\n" + "=" * 60)
print("DATASETS READY!")

In [ ]:
# Cell 5: Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=gfa_collate_fn,
    pin_memory=True,
)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val loader: {len(val_loader)} batches")

In [ ]:
# Cell 6: Batch Remapping Function
def remap_batch(batch):
    """Remap collate output keys to model expected keys."""
    remapped = {}
    for k, v in batch.items():
        # Remove 'brep_' prefix for model compatibility
        if k.startswith('brep_'):
            new_key = k[5:]  # Remove 'brep_' prefix
            remapped[new_key] = v
        else:
            remapped[k] = v
    
    # Split pc_features into local and global
    if 'pc_features' in remapped:
        pc = remapped.pop('pc_features')
        remapped['pc_local_features'] = pc[:, :-1, :]   # (B, N, 1024)
        remapped['pc_global_features'] = pc[:, -1, :]   # (B, 1024)
    
    return remapped

# Test remapping
test_batch = next(iter(train_loader))
test_batch = remap_batch(test_batch)
print("Remapped batch keys:")
for k, v in test_batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

In [ ]:
# Cell 7: Create Model
gc.collect()
torch.cuda.empty_cache()

from clip4cad.models.autobrep_encoder import (
    AutoBrepEncoderConfig,
    AutoBrepStyleEncoder,
    CLIP4CAD_AutoBrep,
    ContrastiveLoss,
)

# Model configuration
config = AutoBrepEncoderConfig(
    d_face=48,
    d_edge=12,
    d=D_MODEL,
    d_proj=D_PROJ,
    num_layers=NUM_LAYERS,
    num_heads=NUM_HEADS,
    dropout=0.1,
    ff_mult=4,
    use_centroids=True,
    use_bfs_levels=True,
    use_normals=True,
    use_areas=True,
)

model = CLIP4CAD_AutoBrep(config).to(device)
print(f"Model parameters: {model.count_parameters():,}")
print(f"Trainable parameters: {model.count_parameters(trainable_only=True):,}")

# Loss function
criterion = ContrastiveLoss(label_smoothing=0.1)

# Scaler for FP16
scaler = GradScaler()

print("\nModel and loss function initialized")

In [ ]:
# Cell 8: Verify Forward Pass
print("Verifying forward pass...")

model.eval()
with torch.no_grad():
    test_batch = remap_batch(next(iter(train_loader)))
    with autocast():
        outputs = model(test_batch)

print("\nOutputs:")
for k, v in outputs.items():
    if isinstance(v, torch.Tensor):
        nan_count = v.isnan().sum().item()
        status = "NaN!" if nan_count > 0 else "OK"
        print(f"  {k}: {v.shape} [{status}]")
    else:
        print(f"  {k}: {v}")

# Test loss
loss, loss_dict = criterion(outputs)
print("\nLoss components:")
for k, v in loss_dict.items():
    print(f"  {k}: {v.item():.4f}")

## Stage 0: Anchor BRep to PC

**Goal:** Make BRep encoder produce meaningful features by aligning to pre-trained PC encoder.

**Why this works:**
- ShapeLLM (PC encoder) is pre-trained and produces meaningful features
- PC and BRep represent the SAME 3D geometry
- BRep encoder has a stable target to learn toward

**Expected:** BRep-PC gap → 0, cosine → 0.7+

In [ ]:
# Cell 10: Training Loop
print("\n" + "="*70)
print("TRAINING: AutoBrep-Style Encoder (Stage 0 - BRep-PC Alignment)")
print("="*70)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = {
    'loss': [], 'gap': [], 'cosine': [], 'r_at_1': []
}

best_cos = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    epoch_metrics = {'gap': [], 'cos': [], 'r1': []}
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")
    
    for batch in pbar:
        batch = remap_batch(batch)
        
        with autocast():
            outputs = model(batch)
            loss, loss_dict = criterion(outputs)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss_dict['total'].item()
        epoch_metrics['gap'].append(loss_dict['gap'].item())
        epoch_metrics['cos'].append(loss_dict['cosine'].item())
        epoch_metrics['r1'].append(loss_dict['r_at_1'].item())
        
        pbar.set_postfix({
            'loss': f"{loss_dict['total'].item():.3f}",
            'gap': f"{loss_dict['gap'].item():.3f}",
            'cos': f"{loss_dict['cosine'].item():.3f}",
            'R@1': f"{loss_dict['r_at_1'].item()*100:.1f}%",
        })
    
    scheduler.step()
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_gap = np.mean(epoch_metrics['gap'])
    avg_cos = np.mean(epoch_metrics['cos'])
    avg_r1 = np.mean(epoch_metrics['r1'])
    
    history['loss'].append(avg_loss)
    history['gap'].append(avg_gap)
    history['cosine'].append(avg_cos)
    history['r_at_1'].append(avg_r1)
    
    # Save best model
    if avg_cos > best_cos:
        best_cos = avg_cos
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_cos': best_cos,
            'config': config,
        }, OUTPUT_DIR / 'checkpoint_best.pt')
        print(f"  ★ New best cosine: {best_cos:.4f}")
    
    print(f"Epoch {epoch}: Loss={avg_loss:.4f}, Gap={avg_gap:.4f}, Cos={avg_cos:.4f}, R@1={avg_r1*100:.1f}%")

print("\n" + "="*70)
print("Training Complete!")
print(f"Best cosine similarity: {best_cos:.4f}")

In [ ]:
# Cell 11: Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Loss
ax = axes[0, 0]
ax.plot(history['loss'])
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.grid(True, alpha=0.3)

# Gap
ax = axes[0, 1]
ax.plot(history['gap'])
ax.axhline(y=0, color='g', linestyle='--', alpha=0.5, label='Target (0)')
ax.set_xlabel('Epoch')
ax.set_ylabel('L2 Gap')
ax.set_title('BRep-PC Gap (should → 0)')
ax.legend()
ax.grid(True, alpha=0.3)

# Cosine
ax = axes[1, 0]
ax.plot(history['cosine'])
ax.axhline(y=1.0, color='g', linestyle='--', alpha=0.5, label='Target (1.0)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cosine Similarity')
ax.set_title('True-Pair Cosine (should → 1.0)')
ax.legend()
ax.grid(True, alpha=0.3)

# R@1
ax = axes[1, 1]
ax.plot([r*100 for r in history['r_at_1']])
ax.axhline(y=100, color='g', linestyle='--', alpha=0.5, label='Target (100%)')
ax.set_xlabel('Epoch')
ax.set_ylabel('R@1 (%)')
ax.set_title('Retrieval Accuracy (should → 100%)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()

In [ ]:
# Cell 12: Save Final Model
torch.save({
    'config': config,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
    'final_metrics': {
        'loss': history['loss'][-1],
        'gap': history['gap'][-1],
        'cosine': history['cosine'][-1],
        'r_at_1': history['r_at_1'][-1],
    },
}, OUTPUT_DIR / 'autobrep_final.pt')

print(f"Final model saved to {OUTPUT_DIR / 'autobrep_final.pt'}")
print(f"Final metrics:")
print(f"  Loss: {history['loss'][-1]:.4f}")
print(f"  Gap: {history['gap'][-1]:.4f}")
print(f"  Cosine: {history['cosine'][-1]:.4f}")
print(f"  R@1: {history['r_at_1'][-1]*100:.1f}%")

In [ ]:
# Cell 13: Validation Evaluation
print("Evaluating on validation set...")

# Load best model
checkpoint = torch.load(OUTPUT_DIR / 'checkpoint_best.pt')
model.load_state_dict(checkpoint['model_state_dict'])

model.eval()
all_z_brep = []
all_z_pc = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Encoding"):
        batch = remap_batch(batch)
        with autocast():
            outputs = model(batch)
        
        all_z_brep.append(outputs['z_brep'].cpu())
        all_z_pc.append(outputs['z_pc'].cpu())

z_brep = torch.cat(all_z_brep, dim=0)
z_pc = torch.cat(all_z_pc, dim=0)

# Normalize
z_brep = F.normalize(z_brep, dim=-1)
z_pc = F.normalize(z_pc, dim=-1)

N = z_brep.shape[0]
print(f"\nValidation set: {N} samples")

# BRep -> PC retrieval
sim_b2p = z_brep @ z_pc.T
ranks = (sim_b2p.argsort(dim=1, descending=True) == torch.arange(N).unsqueeze(1)).float().argmax(dim=1)
r1 = (ranks < 1).float().mean().item() * 100
r5 = (ranks < 5).float().mean().item() * 100
r10 = (ranks < 10).float().mean().item() * 100

# True-pair cosine
true_cos = (z_brep * z_pc).sum(dim=-1).mean().item()

# L2 gap
gap = (z_brep - z_pc).pow(2).sum(-1).sqrt().mean().item()

print("\n" + "="*50)
print("Validation Results (BRep → PC Retrieval):")
print("="*50)
print(f"R@1:  {r1:.1f}%")
print(f"R@5:  {r5:.1f}%")
print(f"R@10: {r10:.1f}%")
print(f"True-pair cosine: {true_cos:.4f}")
print(f"L2 Gap: {gap:.4f}")
print("="*50)

## Next Steps

If Stage 0 training is successful (R@1 > 50%, cosine > 0.5):

1. **Add text modality** (Stage 1): Use the trained BRep encoder as anchor
2. **Three-way contrastive**: Text-BRep-PC alignment
3. **Fine-tuning** (Stage 2): Gap closing + hard negative mining

The key insight is that AutoBrep proves the architecture works on our data.
We just need to train it correctly with good anchoring.